# Simple Economy Model

Agent-based simulation of a minimal macroeconomic model with three agent types:
- **Household** – supplies labour, demands consumption goods, pays income tax.
- **CGFirm** – hires labour, produces and sells goods, pays profit tax and distributes dividends.
- **Government** – collects taxes, pays unemployment benefits, tracks macro aggregates.

In [1]:
import os
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import EcoSimpy

sns.set_theme(style="whitegrid")
pd.options.mode.chained_assignment = None

input_dir  = Path("runs")
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

## Run the Simulation

In [2]:
def main():
    print("Current directory:", os.getcwd())

    config_json_file = "config.json"
    app_dir = os.getcwd()

    sim = EcoSimpy.Simulation(app_dir,
                              config_json_file,
                              clean_run=True)
    sim.execute_simulation()


if __name__ == "__main__":
    main()
else:
    main()

Current directory: /home/rivero/Dropbox/Workspace_Current/Projects/Apps/EcoSim/EcoSimpy_dev/examples/simple_economy_ai
Simulation config file path: /home/rivero/Dropbox/Workspace_Current/Projects/Apps/EcoSim/EcoSimpy_dev/examples/simple_economy_ai/config.json
There is no agent type called Household in this model
The agent vars for this agent type will not be created and the model will probably not work as expected


Scenario: Scenario1 Run nr.: 0:   0%|          | 0/60 [00:00<?, ?it/s]


AttributeError: 'Household' object has no attribute 'reservation_wage'

## Load and Process Output Data

In [ ]:
def load_agent_csv(input_dir, agent_name):
    """Load the first CSV file matching the agent name and drop bookkeeping columns."""
    files = [f for f in os.listdir(input_dir) if agent_name in f and f.endswith(".csv")]
    if files:
        df = pd.read_csv(os.path.join(input_dir, files[0]))
        df = df.drop(columns=[c for c in ["run", "index_no"] if c in df.columns])
        return df
    print(f"No data found for {agent_name}.")
    return pd.DataFrame()


def aggregate_by_step(df):
    """Group by step: mean for numeric columns, sum for boolean columns."""
    if df.empty:
        return pd.DataFrame()
    agg = {
        **{c: "mean" for c in df.select_dtypes(include="number").columns if c != "step"},
        **{c: "sum"  for c in df.select_dtypes(include="bool").columns},
    }
    return df.groupby("step").agg(agg).reset_index()


# --- Load raw data ---
model_name = "simple_economy"

household_raw  = load_agent_csv(input_dir, f"{model_name}_Household")
cgfirm_raw     = load_agent_csv(input_dir, f"{model_name}_CGFirm")
government_raw = load_agent_csv(input_dir, f"{model_name}_Government")

# --- Aggregate by step ---
household_means  = aggregate_by_step(household_raw)
cgfirm_means     = aggregate_by_step(cgfirm_raw)
government_means = aggregate_by_step(government_raw)

# --- Persist aggregates ---
if not household_means.empty:
    household_means.to_csv(output_dir / "household_means.csv", index=False)
if not cgfirm_means.empty:
    cgfirm_means.to_csv(output_dir / "cgfirm_means.csv", index=False)
if not government_means.empty:
    government_means.to_csv(output_dir / "government_means.csv", index=False)

print("Data loading and aggregation complete.")

## Analysis

In [ ]:
def plot_grid(df, title_prefix, ncol=4):
    """Plot every variable in df against the simulation step."""
    if df is None or df.empty:
        print(f"No data to plot for {title_prefix}.")
        return
    cols = [c for c in df.columns if c != "step"]
    n    = len(cols)
    nrow = math.ceil(n / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3 * nrow), squeeze=False)
    for idx, var in enumerate(cols):
        ax = axes[idx // ncol][idx % ncol]
        ax.plot(df["step"], df[var], marker="o", linewidth=1)
        ax.set_title(f"{title_prefix} – {var}", fontsize=10)
        ax.set_xlabel("Step", fontsize=8)
        ax.set_ylabel(var, fontsize=8)
        ax.grid(True, alpha=0.3)
    for idx in range(n, nrow * ncol):
        fig.delaxes(axes[idx // ncol][idx % ncol])
    plt.tight_layout()
    plt.show()

### Household Variables

In [ ]:
plot_grid(household_means, "Household", ncol=4)

### CGFirm Variables

In [ ]:
plot_grid(cgfirm_means, "CGFirm", ncol=4)

### Government Variables

In [ ]:
plot_grid(government_means, "Government", ncol=4)

### Key Macro Indicators

In [ ]:
if not government_means.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for ax, var, label, color in zip(
        axes,
        ["gdp", "unemployment_rate", "price_level"],
        ["GDP", "Unemployment Rate", "Price Level"],
        ["steelblue", "tomato", "seagreen"],
    ):
        if var in government_means.columns:
            ax.plot(government_means["step"], government_means[var],
                    color=color, linewidth=2)
            ax.set_title(label, fontsize=12)
            ax.set_xlabel("Step")
            ax.set_ylabel(label)
            ax.grid(True, alpha=0.3)

    plt.suptitle("Simple Economy – Key Macro Indicators", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

### Household Income vs Consumption Budget

In [ ]:
if not household_means.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for var, label, color in [
        ("income",             "Income",             "steelblue"),
        ("consumption_budget", "Consumption Budget", "tomato"),
        ("savings",            "Savings",            "seagreen"),
        ("wages",              "Wages",              "orange"),
    ]:
        if var in household_means.columns:
            ax.plot(household_means["step"], household_means[var],
                    label=label, color=color, linewidth=2)
    ax.set_title("Household – Income, Consumption and Savings", fontsize=12)
    ax.set_xlabel("Step")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Firm – Revenue, Profit and Inventory

In [ ]:
if not cgfirm_means.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for var, label, color in [
        ("revenue",   "Revenue",   "steelblue"),
        ("profit",    "Profit",    "seagreen"),
        ("inventory", "Inventory", "tomato"),
    ]:
        if var in cgfirm_means.columns:
            ax.plot(cgfirm_means["step"], cgfirm_means[var],
                    label=label, color=color, linewidth=2)
    ax.set_title("CGFirm – Revenue, Profit and Inventory", fontsize=12)
    ax.set_xlabel("Step")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Government Budget

In [ ]:
if not government_means.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for var, label, color in [
        ("tax_revenue",    "Tax Revenue",    "steelblue"),
        ("expenditure",    "Expenditure",    "tomato"),
        ("budget_surplus", "Budget Surplus", "seagreen"),
    ]:
        if var in government_means.columns:
            ax.plot(government_means["step"], government_means[var],
                    label=label, color=color, linewidth=2)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Government – Fiscal Accounts", fontsize=12)
    ax.set_xlabel("Step")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()